# __Dream interpretation RAG__

## __Installs & imports__

In [1]:
! pip install chromadb sentence-transformers

In [53]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

## __Data preparation__

In [3]:
data = pd.read_csv('/Users/lesfabs/code/MartynaC/dreamscope/Notebooks/dream_symbols_clean_v2.csv')
data.head()

,symbol_clean,slug,context,context_clean,meaning_clean
0,m,m,"To see the letter ""M"" in your dream",to see the letter m in your dream,suggests that there is something that you are ...
1,m&m's,mms,To see or eat M&M's in your dream,to see or eat mms in your dream,symbolizes lifes small but sweet rewards more ...
2,m&m's,mms,To dream that you are a giant talking M&M,to dream that you are a giant talking mm,suggests that you feel you are being mislead o...
3,macadamize,macadamize,"To see, walk or travel on a macadamized road i...",to see walk or travel on a macadamized road in...,suggests that you are standing on solid ground...
4,macaroni,macaroni,To dream that you are eating macaroni,to dream that you are eating macaroni,symbolizes comfort and ease the dream may be t...


In [4]:
data.context_clean.isnull().sum()

np.int64(7)

In [5]:
# Replace NaNs in  context_clean with the value of the slug column 
data.loc[data['context_clean'].isnull(), 'context_clean'] = data.loc[data['context_clean'].isnull(), 'slug']
data.context_clean.isnull().sum()

np.int64(0)

In [6]:
# Create chunks
data['chunk'] = data['context_clean'] + " " + data['meaning_clean']
data.head()

,symbol_clean,slug,context,context_clean,meaning_clean,chunk
0,m,m,"To see the letter ""M"" in your dream",to see the letter m in your dream,suggests that there is something that you are ...,to see the letter m in your dream suggests tha...
1,m&m's,mms,To see or eat M&M's in your dream,to see or eat mms in your dream,symbolizes lifes small but sweet rewards more ...,to see or eat mms in your dream symbolizes lif...
2,m&m's,mms,To dream that you are a giant talking M&M,to dream that you are a giant talking mm,suggests that you feel you are being mislead o...,to dream that you are a giant talking mm sugge...
3,macadamize,macadamize,"To see, walk or travel on a macadamized road i...",to see walk or travel on a macadamized road in...,suggests that you are standing on solid ground...,to see walk or travel on a macadamized road in...
4,macaroni,macaroni,To dream that you are eating macaroni,to dream that you are eating macaroni,symbolizes comfort and ease the dream may be t...,to dream that you are eating macaroni symboliz...


## __MVP__

### __Embedding__

In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# One chunk
tryout_vector = model.encode(data['chunk'][0])
tryout_vector.shape

(384,)

In [9]:
# All chunks

embeddings = model.encode(data['chunk'], show_progress_bar=True)


Batches:   0%|          | 0/244 [00:00<?, ?it/s]

In [10]:
embeddings.shape

(7798, 384)

### __Setting up and querying collection__

In [11]:
# Initializing collection - slightly different from the method seen in previous challenges using LangChain 
# -> Here it's native ChromaDB, without an intermediary framework

client = chromadb.Client()
collection = client.create_collection("dream_symbols")

In [18]:
# Adding and indexing chunks

collection.add(
    documents=data['chunk'].tolist()[:5000],
    embeddings=embeddings[:5000],
    ids=[str(i) for i in range(5000)]
)



In [19]:
# We do it in two batches because of the limited batch size

collection.add(
    documents=data['chunk'].tolist()[5000:],
    embeddings=embeddings[5000:],
    ids=[str(i) for i in range(5000, 7798)]
)


In [20]:
collection.count()

7798

In [44]:
dreamer_text = "I was in a deep dark forest. I got lost. Then I met an accountant who was eating M&M's. I got scared and ran, only to be faced with an aardvark"

In [45]:
searched_vector = model.encode(dreamer_text)
searched_vector.shape

(384,)

In [46]:
result=collection.query(
    searched_vector,
    n_results=3
)
result['documents']

[['to dream that you are lost in a forest indicates that you are searching through your subconscious for a better understanding of yourself',
  'to dream that you have been ambushed forewarns of a danger lurking near you your situation has taken an unanticipated turn for the worse you have been prevented from reaching your goals or destination',
  'to see someone affected with gangrene in your dream foretells of grief and loss']]

### __Compiling query process in a dedicated function__ 

In [47]:
# Compile code in a dedicated function

def interpret(dream='',n_results=3):
    searched_vector = model.encode(dream)
    result=collection.query(
    searched_vector,
    n_results=n_results
    )
    return result['documents']

interpret(dreamer_text)

[['to dream that you are lost in a forest indicates that you are searching through your subconscious for a better understanding of yourself',
  'to dream that you have been ambushed forewarns of a danger lurking near you your situation has taken an unanticipated turn for the worse you have been prevented from reaching your goals or destination',
  'to see someone affected with gangrene in your dream foretells of grief and loss']]

## __Trying to fine-tune the querying process__

### __Extracting keywords__

In [54]:
# Initialize Model

tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct')

# Initialize pipeline

pipe = pipeline( 
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    )


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [55]:
# Define function to extract keywords

def extract_keywords(dreamer_text, n_keywords):
    # define prompt template
    prompt = f'You process a text input by extracting the {n_keywords} most significant keywords. Expected format : a Python list of singular words, without any additional text'
    messages = [
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': dreamer_text}
    ]

    # query model
    output = pipe(messages) 
    return output[0]['generated_text'][-1]['content']
    

In [60]:
extract_keywords(dreamer_text, 8)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Task was destroyed but it is pending!
task: <Task pending name='Task-256' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at /Users/lesfabs/.pyenv/versions/3.10.6/envs/dreamscope/lib/python3.10/site-packages/ipykernel/utils.py:76> wait_for=<Task pending name='Task-257' coro=<_async_in_context.<locals>.preserve_context() running at /Users/lesfabs/.pyenv/versions/3.10.6/envs/dreamscope/lib/python3.10/site-packages/ipykernel/utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/lesfabs/.pyenv/versions/3.10.6/envs/dreamscope/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/lesfabs/.pyenv/versions/3.10.6/envs/dreamscope/lib/python3.10/site-packages/torch/nn/mod

" ['forest', 'lost', 'accountant', 'M&M\\'s','scared', 'ran', 'aardvark', 'dark']"